In [1]:
import sys
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

In [2]:
os.chdir("../")
%pwd

'/home/aditya/Desktop/Agentic/FinancialMarketEventRouter'

In [3]:
from src.config.configuration import ConfigurationManager

In [4]:
config_manager = ConfigurationManager(
    config_filepath="config/config.yaml",
    params_filepath="params.yaml"
)

In [5]:
llm_config = config_manager.get_llm_config()

In [6]:
llm = ChatOpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url=llm_config.base_url,
    model=llm_config.model_name,
    temperature=llm_config.temperature,
    max_tokens=llm_config.max_tokens
)

In [7]:
print(f"Initialized LLM: {llm_config.model_name}")

Initialized LLM: gemini-3.5-flash


In [8]:
import yfinance as yf
from pydantic import BaseModel, Field
from langchain_core.tools import tool

In [9]:
class SearchInput(BaseModel):
    query: str = Field(description="The stock ticker symbol to search for (e.g., AAPL, MSFT, NVDA).")

In [10]:
@tool("financial_search", args_schema=SearchInput)
def financial_search_tool(query: str) -> str:
    """Fetches real-time financial headlines for a given stock ticker."""
    ticker = yf.Ticker(query)
    news = ticker.news
    
    if not news:
        return f"No recent news found for ticker {query}."
    
    formatted_news = []
    for article in news[:3]:
        title = article.get('title') or article.get('content', {}).get('title', 'Headline Unavailable')
        publisher = article.get('publisher') or article.get('content', {}).get('provider', {}).get('displayName', 'Unknown Publisher')
        formatted_news.append(f"- {title} ({publisher})")

    return f"Latest headlines for {query}:\n" + "\n".join(formatted_news)

In [11]:
tools = [financial_search_tool]
llm_with_tools = llm.bind_tools(tools)

In [12]:
from typing import TypedDict, Annotated, Sequence, Literal
import operator
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage

In [23]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    route_action: str

def router_node(state: AgentState):
    """Evaluates the user query and determines whether to invoke tools or synthesize directly."""
    response = llm_with_tools.invoke(state["messages"])
    action = "tool" if response.tool_calls else "synthesize"
    return {"messages": [response], "route_action": action}

def synthesizer_node(state: AgentState):
    """Reads injected context from the state and generates the final response."""
    messages = state["messages"]
    user_query = messages[0].content

    tool_content = ""
    for msg in messages:
        if msg.type == "tool":
            tool_content = msg.content
            break
            
    system_prompt = SystemMessage(
        content="You are a financial market analyst. Synthesize the provided context and news into a concise, accurate market update."
    )

    synthesis_messages = [
        system_prompt,
        HumanMessage(content=f"Query: {user_query}\n\nContext from Tool:\n{tool_content}")
    ]
    
    response = llm.invoke(synthesis_messages)
    return {"messages": [response]}

In [24]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

In [25]:
workflow = StateGraph(AgentState)

In [26]:
workflow.add_node("router", router_node)
workflow.add_node("tool", ToolNode(tools))
workflow.add_node("synthesize", synthesizer_node)

In [27]:
workflow.add_edge(START, "router")

In [28]:
def check_route(state: AgentState) -> Literal["tool", "synthesize"]:
    return "tool" if state["route_action"] == "tool" else "synthesize"

In [29]:
workflow.add_conditional_edges(
    "router",
    check_route,
    {
        "tool": "tool",
        "synthesize": "synthesize"
    }
)

In [30]:
workflow.add_edge("tool", "synthesize")
workflow.add_edge("synthesize", END)

graph = workflow.compile()
print("Graph compiled successfully.")

Graph compiled successfully.


In [31]:
test_query = "What is the latest news regarding Apple (AAPL) stock?"
initial_state = {
    "messages": [HumanMessage(content=test_query)],
    "route_action": ""
}

final_output = graph.invoke(initial_state)

print("--- FINAL RESPONSE ---")
print(final_output["messages"][-1].content)

--- FINAL RESPONSE ---
Based on the latest market headlines, the key focus for Apple (AAPL) revolves around its **upcoming product launch event**. 

Financial analysts, including those at Zacks, are currently evaluating whether the stock is a strategic "buy" ahead of this major event, which typically serves as a significant catalyst for the company's shares. 

Additionally, Apple remains a central point of discussion in broader market trends concerning high-yield Nasdaq-100 companies and the ongoing reshuffling of major ETFs toward prominent AI-focused stocks.
